  pip install \
    "langchain-google-vertexai>=3.0,<4" \
    "langgraph>=0.2,<2" \
    "langchain-core>=0.3" \
    "google-cloud-bigquery>=3.11"

# Capital Allocation Analyst — LangGraph + Gemini 2.5 (Vertex AI) + BigQuery

Same capital-allocation agent as `financial_analyst_adk.ipynb`, but the agentic
logic is now implemented with **LangGraph** instead of Google ADK. The model is
still **Gemini 2.5 Pro served through Vertex AI** (`ChatVertexAI`), data still
comes **read-only from BigQuery**, and the financial-math `calc` tool is
preserved for NPV / IRR / payback / etc.

What's different from the ADK notebook:

| Concern | `financial_analyst_adk.ipynb` (ADK) | this notebook (LangGraph) |
|---|---|---|
| Agent loop | ADK `LlmAgent` + `Runner` (hidden) | **explicit `StateGraph`**: agent ⇄ tools |
| Tool-calling | ADK toolset protocol | `llm.bind_tools(...)` + `ToolNode` |
| Memory | ADK sessions | LangGraph `MemorySaver` checkpointer (per `thread_id`) |
| BigQuery | ADK first-party toolset | custom read-only tools on `google-cloud-bigquery` |
| Model | `ChatVertexAI("gemini-2.5-pro")` via Vertex AI | same |

The graph is the classic ReAct shape, written out explicitly so the
perception→action loop is visible:

```
        +-------+      tool calls?     +-------+
 START -> agent | -------------------> | tools |
        +-------+ \                    +-------+
            ^      \  no tool calls         |
            |       ------> END             |
            +-------------------------------+
```

**Safety posture:** BigQuery access is **read-only** — queries must be
`SELECT`/`WITH`, are **dry-run first**, and are **hard-capped** by
`maximum_bytes_billed`. Point it at a financial institution's warehouse only
with least-privilege credentials.

---

### Setup

```bash
pip install langgraph langchain-google-vertexai langchain-core google-cloud-bigquery

# Application Default Credentials drive BOTH Vertex AI and BigQuery:
gcloud auth application-default login
#   ...or: export GOOGLE_APPLICATION_CREDENTIALS=/path/to/service-account.json

export GOOGLE_CLOUD_PROJECT=your-gcp-project
export GOOGLE_CLOUD_LOCATION=us-central1
```


In [ ]:
# Optional: install dependencies (uncomment on first run)
# %pip install -q langgraph langchain-google-vertexai langchain-core google-cloud-bigquery

In [ ]:
"""
Imports + configuration. All knobs live here.
"""
from __future__ import annotations

import json
import logging
import math
import os
import re

PROJECT     = os.environ.get("GOOGLE_CLOUD_PROJECT", "")          # set this
LOCATION    = os.environ.get("GOOGLE_CLOUD_LOCATION", "us-central1")  # Vertex region
MODEL       = os.environ.get("FIN_AGENT_MODEL", "gemini-2.5-pro")

# BigQuery
BQ_LOCATION         = os.environ.get("BQ_LOCATION", "US")
BQ_MAX_BYTES_BILLED = int(os.environ.get("BQ_MAX_BYTES_BILLED", str(1 << 30)))  # 1 GiB
BQ_MAX_RESULT_ROWS  = int(os.environ.get("BQ_MAX_RESULT_ROWS", "200"))

# Gemini 2.5 thinking budget (tokens). -1 = let the model decide dynamically.
THINKING_BUDGET = int(os.environ.get("FIN_THINKING_BUDGET", "-1"))

logging.basicConfig(
    level=os.environ.get("AGENT_LOG_LEVEL", "INFO").upper(),
    format="[%(levelname)s] %(name)s | %(message)s",
)
log = logging.getLogger("fin_lg")
log.info("model=%s vertex_location=%s bq_location=%s", MODEL, LOCATION, BQ_LOCATION)

## `calc` — sandboxed financial-math tool

Ported from the original notebook and wrapped as a LangChain tool. The model
passes a Python expression; we `eval` it with `__builtins__` stripped and a
curated namespace (every `math.*` function plus Excel-style finance helpers).
LangGraph reads the type hints + docstring to build the tool schema.

In [ ]:
from langchain_core.tools import tool


def _npv(rate, cashflows):
    return sum(cf / (1.0 + rate) ** i for i, cf in enumerate(cashflows))


def _irr(cashflows, guess: float = 0.1):
    lo, hi = -0.9999, 10.0
    f_lo, f_hi = _npv(lo, cashflows), _npv(hi, cashflows)
    if f_lo * f_hi <= 0:                       # bracketed -> bisection
        for _ in range(200):
            mid = (lo + hi) / 2.0
            f_mid = _npv(mid, cashflows)
            if abs(f_mid) < 1e-10:
                return mid
            if f_lo * f_mid < 0:
                hi = mid
            else:
                lo, f_lo = mid, f_mid
        return (lo + hi) / 2.0
    r = guess                                  # not bracketed -> Newton
    for _ in range(100):
        base = _npv(r, cashflows)
        deriv = (_npv(r + 1e-6, cashflows) - base) / 1e-6
        if deriv == 0:
            break
        step = base / deriv
        r -= step
        if abs(step) < 1e-10:
            return r
    return r


def _fv(rate, nper, pmt, pv=0.0):
    if rate == 0:
        return -(pv + pmt * nper)
    return -(pv * (1 + rate) ** nper + pmt * ((1 + rate) ** nper - 1) / rate)


def _pv(rate, nper, pmt, fv=0.0):
    if rate == 0:
        return -(fv + pmt * nper)
    return -(fv + pmt * ((1 + rate) ** nper - 1) / rate) / (1 + rate) ** nper


def _pmt(rate, nper, pv, fv=0.0):
    if rate == 0:
        return -(pv + fv) / nper
    return -(pv * (1 + rate) ** nper + fv) * rate / ((1 + rate) ** nper - 1)


@tool
def calc(expression: str) -> str:
    """Evaluate a Python/finance math expression and return the numeric result.

    Use this for ALL arithmetic instead of computing in your head. The namespace
    exposes every `math.*` function plus Excel-style finance helpers:
      npv(rate, cashflows)         net present value; cashflows[0] is t=0
      irr(cashflows)               internal rate of return
      fv(rate, nper, pmt, pv=0)    future value of an annuity
      pv(rate, nper, pmt, fv=0)    present value of an annuity
      pmt(rate, nper, pv, fv=0)    periodic payment for a loan/annuity

    Examples:
      calc("npv(0.10, [-1000, 300, 400, 500, 600])")
      calc("irr([-1000, 300, 400, 500, 600])")
      calc("pmt(0.05/12, 360, 400000)")
    """
    log.info("[calc] %s", expression[:160])
    ns = {k: getattr(math, k) for k in dir(math) if not k.startswith("_")}
    ns.update({
        "abs": abs, "round": round, "min": min, "max": max, "sum": sum,
        "len": len, "pow": pow, "range": range, "sorted": sorted,
        "npv": _npv, "irr": _irr, "fv": _fv, "pv": _pv, "pmt": _pmt,
    })
    try:
        return str(eval(expression, {"__builtins__": {}}, ns))  # sandboxed
    except Exception as e:
        return f"Error: {e}  (expression: {expression!r})"


# quick self-check (no model / no GCP needed)
print(calc.invoke({"expression": "npv(0.10, [-1000, 300, 400, 500, 600])"}))
print(calc.invoke({"expression": "pmt(0.05/12, 360, 400000)"}))

## Read-only BigQuery tools

Built directly on the `google-cloud-bigquery` client so we fully control the
safety posture. Free metadata tools (`bq_list_datasets`, `bq_list_tables`,
`bq_table_schema`) do the orientation; `bq_query` runs read-only SQL only,
**dry-runs first**, refuses anything scanning more than `BQ_MAX_BYTES_BILLED`,
and sets `maximum_bytes_billed` as a hard stop. Results are capped to
`BQ_MAX_RESULT_ROWS`.

In [ ]:
from google.cloud import bigquery

_bq = None

def bq_client() -> "bigquery.Client":
    global _bq
    if _bq is None:
        _bq = bigquery.Client(project=PROJECT or None, location=BQ_LOCATION)
    return _bq


_READONLY_RE  = re.compile(r"^\s*(SELECT|WITH)\b", re.I)
_FORBIDDEN_RE = re.compile(
    r"\b(INSERT|UPDATE|DELETE|MERGE|DROP|CREATE|ALTER|TRUNCATE|GRANT|"
    r"REVOKE|CALL|EXPORT|LOAD)\b", re.I)


def _guarded_query(sql: str):
    """Validate (read-only), dry-run for cost, then execute capped. Returns
    (rows, error)."""
    if not _READONLY_RE.match(sql) or _FORBIDDEN_RE.search(sql):
        return None, "Refused: only read-only SELECT/WITH queries are allowed."
    try:
        dry = bq_client().query(
            sql, job_config=bigquery.QueryJobConfig(dry_run=True, use_query_cache=False))
        scanned = dry.total_bytes_processed or 0
        if scanned > BQ_MAX_BYTES_BILLED:
            return None, (f"Refused: query would scan {scanned/1e9:.2f} GB, over the "
                          f"{BQ_MAX_BYTES_BILLED/1e9:.2f} GB cap. Add filters/limits.")
        job = bq_client().query(
            sql, job_config=bigquery.QueryJobConfig(
                maximum_bytes_billed=BQ_MAX_BYTES_BILLED))
        rows = [dict(r) for r in job.result(max_results=BQ_MAX_RESULT_ROWS)]
        return rows, None
    except Exception as e:
        return None, f"Error: {e}"


@tool
def bq_list_datasets() -> str:
    """List dataset IDs available in the BigQuery project."""
    try:
        ids = [d.dataset_id for d in bq_client().list_datasets()]
        return json.dumps(ids)
    except Exception as e:
        return f"Error: {e}"


@tool
def bq_list_tables(dataset: str) -> str:
    """List table IDs in a dataset. Args: dataset (the dataset id)."""
    try:
        ids = [t.table_id for t in bq_client().list_tables(dataset)]
        return json.dumps(ids)
    except Exception as e:
        return f"Error: {e}"


@tool
def bq_table_schema(dataset: str, table: str) -> str:
    """Return columns/types/descriptions, row count, size, and partition/cluster
    info for one table. Use this to find grain and join keys before writing SQL.
    Args: dataset, table."""
    try:
        t = bq_client().get_table(f"{dataset}.{table}")
        cols = [{"name": f.name, "type": f.field_type, "mode": f.mode,
                 "description": f.description} for f in t.schema]
        info = {
            "full_id": t.full_table_id,
            "rows": t.num_rows,
            "size_mb": round((t.num_bytes or 0) / 1e6, 2),
            "partitioning": (t.time_partitioning.field
                             if t.time_partitioning else None),
            "clustering": t.clustering_fields,
            "columns": cols,
        }
        return json.dumps(info, default=str)
    except Exception as e:
        return f"Error: {e}"


@tool
def bq_query(sql: str) -> str:
    """Run a READ-ONLY BigQuery SQL query (SELECT/WITH only). It is dry-run first
    and refused if it would scan more than the byte cap; results are row-capped.
    Use fully-qualified `dataset.table`, select only needed columns, and aggregate
    in SQL. Returns rows as JSON. Args: sql."""
    rows, err = _guarded_query(sql)
    if err:
        return err
    return json.dumps(rows, default=str)


TOOLS = [bq_list_datasets, bq_list_tables, bq_table_schema, bq_query, calc]
log.info("tools ready: %s", [t.name for t in TOOLS])

## System instruction

The capital-allocation analyst persona: orient → plan → query cost-aware →
compute investment metrics with `calc` → rank & recommend an allocation →
explain business meaning/caveats → cite tables.

In [ ]:
INSTRUCTION = """\
You are a senior analyst supporting a CAPITAL ALLOCATION process. You have
direct, READ-ONLY access to a BigQuery data warehouse and a precise
financial-math calculator. Your analysis informs how limited capital is
deployed across competing projects, business units, segments, or assets.

Work like an investment-committee analyst, not a chatbot:

1. ORIENT. When you don't yet know the data, discover it first. Use the
   BigQuery tools (bq_list_datasets, bq_list_tables, bq_table_schema) to map
   datasets, tables and schemas BEFORE writing any SQL. Never invent table or
   column names. Find the tables holding capital-allocation inputs: cash flows,
   revenue/cost, invested capital, asset/segment identifiers, dates, and any
   risk-weighting or cost-of-capital references.

2. PLAN. State the candidate units (projects/segments/assets) you will compare,
   the tables and grain of each, and how you will join them. Confirm join keys
   from the schema, not from assumption.

3. QUERY CAREFULLY with bq_query. It is read-only (SELECT/WITH only), dry-run
   and byte-capped. Select only needed columns, filter by partition/date, and
   aggregate in SQL rather than pulling raw rows. Validate assumptions with
   small probing queries before large aggregations.

4. COMPUTE EXACTLY with `calc`. Do not do arithmetic in your head. For each
   candidate derive the metrics the question calls for, e.g.:
     - NPV at the relevant discount / hurdle rate      -> calc("npv(...)")
     - IRR of the cash-flow stream                     -> calc("irr(...)")
     - payback period, ROIC / RAROC, profitability index
     - return spread vs. cost of capital (EVA-style)
   Be explicit about the discount / hurdle rate you use and where it came from
   (a warehouse reference table if available, otherwise a stated assumption).

5. RANK & RECOMMEND. Compare candidates on a consistent basis, rank them, and
   give a clear allocation recommendation. Respect any stated capital budget or
   constraints; if a budget is given, allocate to maximize value within it and
   note what is funded vs. deferred and why.

6. EXPLAIN BUSINESS MEANING & CAVEATS. Describe what each table represents and
   the business rules you observed (status filters, currency handling,
   soft-deletes, capitalized vs. expensed items, period cut-offs). Flag data
   quality, coverage, or comparability issues that could change the ranking.

7. CITE. Reference the exact `dataset.table` and columns behind every figure so
   the committee can reproduce it.

If a request is ambiguous (discount rate, horizon, or capital budget
unspecified), state the assumption you're making and proceed; surface anything
that would materially change the allocation decision.
"""
print(INSTRUCTION[:200], "...")

## Model — Gemini 2.5 Pro via Vertex AI, with tools bound

`ChatVertexAI` routes to Gemini on Vertex AI using Application Default
Credentials. We enable Gemini 2.5 "thinking" and bind the tool set so the model
can emit tool calls that LangGraph will execute.

In [ ]:
from langchain_google_vertexai import ChatVertexAI

llm = ChatVertexAI(
    model=MODEL,
    project=PROJECT or None,
    location=LOCATION,
    temperature=0.2,
    thinking_budget=THINKING_BUDGET,   # Gemini 2.5 reasoning budget (-1 = auto)
)
llm_with_tools = llm.bind_tools(TOOLS)
log.info("ChatVertexAI ready: %s", MODEL)

## The LangGraph agent loop

An explicit `StateGraph` over `MessagesState`:

- **agent** node calls the tool-bound LLM (system prompt prepended each turn).
- **tools** node (`ToolNode`) runs whatever tool calls the LLM emitted.
- `tools_condition` routes agent→tools while there are tool calls, else →END.
- A `MemorySaver` checkpointer persists state per `thread_id`, so follow-up
  questions on the same thread keep context.

In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage
from langgraph.graph import StateGraph, MessagesState, START
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.checkpoint.memory import MemorySaver

_SYS = SystemMessage(content=INSTRUCTION)


def agent_node(state: MessagesState) -> dict:
    """Perception->action: prepend the system prompt and let the model decide
    whether to answer or call a tool."""
    response = llm_with_tools.invoke([_SYS] + state["messages"])
    return {"messages": [response]}


builder = StateGraph(MessagesState)
builder.add_node("agent", agent_node)
builder.add_node("tools", ToolNode(TOOLS))
builder.add_edge(START, "agent")
builder.add_conditional_edges("agent", tools_condition)   # -> "tools" or END
builder.add_edge("tools", "agent")

graph = builder.compile(checkpointer=MemorySaver())
log.info("graph compiled")
# Optional: visualise -> graph.get_graph().print_ascii()

## Drive it

`run()` streams the graph, logging each tool call as it happens and returning
the final answer. Reuse a `thread_id` across calls for multi-turn follow-ups
(the `MemorySaver` keeps the conversation state).

Set `GOOGLE_CLOUD_PROJECT` and authenticate first.

In [ ]:
def run(question: str, thread_id: str = "default") -> str:
    config = {"configurable": {"thread_id": thread_id}}
    final = None
    for event in graph.stream(
        {"messages": [HumanMessage(content=question)]}, config, stream_mode="values"
    ):
        msg = event["messages"][-1]
        for tc in getattr(msg, "tool_calls", None) or []:
            log.info("  -> tool call: %s(%s)", tc["name"], tc["args"])
        final = msg
    return final.content if final is not None else ""


log.info("run() ready")

In [ ]:
# answer = run(
#     "List the available datasets and tables, then identify the tables holding "
#     "project/segment cash flows and invested capital. Tell me their grain and "
#     "the keys that join them.",
#     thread_id="capalloc-2026q2")
# print(answer)

In [ ]:
# Follow-up on the SAME thread_id (context carries over):
#
# answer = run(
#     "For each candidate project, pull the annual cash flows and compute NPV at "
#     "a 10% hurdle rate, IRR, and payback using calc. Rank them and recommend "
#     "how to allocate a 500M capital budget, noting what is funded vs. deferred.",
#     thread_id="capalloc-2026q2")
# print(answer)